In [1]:
!pip install nltk spacy word2number
!python -m spacy download en_core_web_sm
%pip uninstall -y torchvision torchaudio torchtext
%pip install -q -U "transformers>=4.48.0" "accelerate>=0.26.0" datasets scikit-learn pandas nltk seaborn matplotlib


  Preparing metadata (setup.py) ... done
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5568 sha256=51c869d133d0358f6655f1c9936a8abdb15da36fc311b14d84b42c9fd2137457
  Stored in directory: /root/.cache/pip/wheels/5b/79/fb/d25928e599c7e11fe4e00d32048cd74933f34a74c633d2aea6
Successfully built word2number
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 51.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfu

In [ ]:
# --- Hybrid ModernBERT + ELECTRA WITHOUT Augmentation ---
import os
import random
import re

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

os.environ["WANDB_DISABLED"] = "true"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DATA_URL = 'https://raw.githubusercontent.com/egbertbritannia-cpu/rbl-requirements-classification/main/data/exp/promise_exp.csv'
df = pd.read_csv(DATA_URL)
df = df[['RequirementText', 'class']].dropna().reset_index(drop=True)
df['model_text'] = df['RequirementText'].astype(str)
df = df[df['class'].map(df['class'].value_counts()) > 1].reset_index(drop=True)

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['class'])
train_df, val_test_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, stratify=val_test_df['label'], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_used_df = train_df.copy()
print("Train distribution (no augmentation):", dict(train_used_df['class'].value_counts()))
print("Validation distribution:", dict(val_df['class'].value_counts()))
print("Test distribution:", dict(test_df['class'].value_counts()))

mb_tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
el_tok = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

def tokenize(tokenizer, texts):
    return tokenizer(texts, truncation=True, padding="max_length", max_length=128, return_token_type_ids=False)

class HybridDataset(Dataset):
    def __init__(self, texts, labels):
        self.mb = tokenize(mb_tok, texts)
        self.el = tokenize(el_tok, texts)
        self.labels = [int(label) for label in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "mb_input_ids": torch.tensor(self.mb["input_ids"][idx], dtype=torch.long),
            "mb_attention_mask": torch.tensor(self.mb["attention_mask"][idx], dtype=torch.long),
            "el_input_ids": torch.tensor(self.el["input_ids"][idx], dtype=torch.long),
            "el_attention_mask": torch.tensor(self.el["attention_mask"][idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = HybridDataset(train_used_df["model_text"].astype(str).tolist(), train_used_df["label"].tolist())
val_ds = HybridDataset(val_df["model_text"].astype(str).tolist(), val_df["label"].tolist())
test_ds = HybridDataset(test_df["model_text"].astype(str).tolist(), test_df["label"].tolist())


def freeze_lower_encoder_layers(encoder):
    for layer_path, layers in [
        ("encoder.layer", getattr(getattr(encoder, "encoder", None), "layer", None)),
        ("layers", getattr(encoder, "layers", None)),
        ("model.layers", getattr(getattr(encoder, "model", None), "layers", None)),
    ]:
        if layers is not None and len(layers) >= 8:
            # Freeze lower transformer layers to reduce overfitting and computation on short requirement texts.
            for layer in layers[:8]:
                for param in layer.parameters():
                    param.requires_grad = False
            print(f"Frozen lower transformer layers: {layer_path}[0:8]")
            return
    raise ValueError("Could not find transformer layers to freeze.")


def report_trainable_and_frozen_parameters(model):
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    frozen_params = sum(param.numel() for param in model.parameters() if not param.requires_grad)
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters: {frozen_params:,}")

class GatedFusion(nn.Module):
    def __init__(self, n_labels, weights=None):
        super().__init__()
        self.mb = AutoModel.from_pretrained("answerdotai/ModernBERT-base")
        self.el = AutoModel.from_pretrained("google/electra-base-discriminator")
        freeze_lower_encoder_layers(self.mb)
        freeze_lower_encoder_layers(self.el)
        self.gate = nn.Linear(768 * 2, 768)
        self.dropout = nn.Dropout(0.3)
        self.clf = nn.Linear(768, n_labels)
        report_trainable_and_frozen_parameters(self)
        if weights is not None:
            self.register_buffer("weights", weights)
        else:
            self.weights = None

    def forward(self, mb_input_ids=None, mb_attention_mask=None, el_input_ids=None, el_attention_mask=None, labels=None, **kwargs):
        mb_outputs = self.mb(input_ids=mb_input_ids, attention_mask=mb_attention_mask)
        el_outputs = self.el(input_ids=el_input_ids, attention_mask=el_attention_mask)
        h_mb = mb_outputs.last_hidden_state[:, 0, :]
        h_el = el_outputs.last_hidden_state[:, 0, :]
        h_concat = torch.cat([h_mb, h_el], dim=-1)
        g = torch.sigmoid(self.gate(h_concat))
        h_fused = g * h_mb + (1 - g) * h_el
        logits = self.clf(self.dropout(h_fused))
        loss = F.cross_entropy(logits, labels, weight=self.weights) if labels is not None else None
        return {"loss": loss, "logits": logits}

num_labels = len(label_encoder.classes_)
weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_labels), y=train_used_df["label"].astype(int))
weights = torch.tensor(weights, dtype=torch.float)
model = GatedFusion(n_labels=num_labels, weights=weights)

for param in model.parameters():
    if not param.is_contiguous():
        param.data = param.data.contiguous()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": p_w,
        "recall": r_w,
        "f1": f1_w,
        "precision_macro": p_m,
        "recall_macro": r_m,
        "macro_F1": f1_m,
    }

training_args = TrainingArguments(
    output_dir="./modernbert_electra_gated_fusion_" + ("with_augmentation" if False else "without_augmentation"),
    num_train_epochs=30,
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    weight_decay=0.02,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_F1",
    greater_is_better=True,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()

print("Validation Results:")
print(trainer.evaluate(val_ds))
print("\nTesting Results:")
test_results = trainer.predict(test_ds)
test_preds = np.argmax(test_results.predictions, axis=-1)
print(classification_report(test_df["label"].astype(int).tolist(), test_preds, target_names=label_encoder.classes_, zero_division=0))


In [ ]:
# --- Hybrid ModernBERT + ELECTRA WITH Augmentation ---
import os
import random
import re

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
import nltk
from nltk.corpus import stopwords, wordnet

os.environ["WANDB_DISABLED"] = "true"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
KEEP_WORDS = {'not', 'no', 'shall', 'must', 'should', 'will', 'cannot', 'never', 'only'}
stop_words = set(stopwords.words('english')) - KEEP_WORDS

def get_synonyms(word):
    clean_word = re.sub(r'[^a-zA-Z]', '', str(word).lower())
    if not clean_word:
        return []
    synonyms = set()
    for syn in wordnet.synsets(clean_word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ").lower()
            if synonym != clean_word:
                synonyms.add(synonym)
    return list(synonyms)

def synonym_replacement(text, n=1):
    words = str(text).split()
    new_words = words.copy()
    random_word_list = list(set([
        word for word in words
        if re.sub(r'[^a-zA-Z]', '', word).lower() not in stop_words and len(word) > 3
    ]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word)
        if len(synonyms) >= 1:
            synonym = random.choice(synonyms)
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return ' '.join(new_words)

def phrase_insertion(text, n=1):
    phrases = ["the system shall", "must be able to", "should ensure that"]
    words = str(text).split()
    if not words:
        return str(text)
    for _ in range(n):
        words.insert(random.randint(0, len(words) - 1), random.choice(phrases))
    return ' '.join(words)

def random_deletion(text, p=0.1):
    words = str(text).split()
    if len(words) <= 1:
        return str(text)
    remaining = [word for word in words if random.random() > p]
    return ' '.join(remaining) if remaining else random.choice(words)

def random_swap(text, n=1):
    words = str(text).split()
    if len(words) < 2:
        return str(text)
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(words)), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]
    return ' '.join(words)

def augment_text(text):
    aug_text = synonym_replacement(text, 1)
    aug_text = phrase_insertion(aug_text, 1)
    aug_text = random_swap(aug_text, 1)
    aug_text = random_deletion(aug_text, p=0.1)
    return ' '.join(str(aug_text).split())

DATA_URL = 'https://raw.githubusercontent.com/egbertbritannia-cpu/rbl-requirements-classification/main/data/exp/promise_exp.csv'
df = pd.read_csv(DATA_URL)
df = df[['RequirementText', 'class']].dropna().reset_index(drop=True)
df['model_text'] = df['RequirementText'].astype(str)
df = df[df['class'].map(df['class'].value_counts()) > 1].reset_index(drop=True)

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['class'])
train_df, val_test_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, stratify=val_test_df['label'], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

augmentation_targets = {
    "FT": 200,
    "L": 200,
    "MN": 200,
    "A": 200,
    "LF": 200,
    "SC": 200,
    "PO": 200,
    "PE": 200,
    "O": 200,
    "US": 200,
    "SE": 100,
}
augmented = []
for cls, target in augmentation_targets.items():
    cls_rows = train_df[train_df['class'] == cls]
    needed = max(0, target - len(cls_rows))
    if len(cls_rows) == 0 or needed == 0:
        continue
    for _ in range(needed):
        row = cls_rows.sample(n=1, random_state=random.randint(0, 10**9)).iloc[0]
        aug_text = augment_text(row['RequirementText'])
        augmented.append({'model_text': aug_text, 'RequirementText': aug_text, 'class': cls, 'label': row['label']})
train_used_df = pd.concat([train_df, pd.DataFrame(augmented)], ignore_index=True) if len(augmented) > 0 else train_df.copy()
train_used_df = train_used_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Original train distribution:", dict(train_df['class'].value_counts()))
print("Augmented train distribution:", dict(train_used_df['class'].value_counts()))
print("Validation distribution:", dict(val_df['class'].value_counts()))
print("Test distribution:", dict(test_df['class'].value_counts()))

mb_tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
el_tok = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

def tokenize(tokenizer, texts):
    return tokenizer(texts, truncation=True, padding="max_length", max_length=128, return_token_type_ids=False)

class HybridDataset(Dataset):
    def __init__(self, texts, labels):
        self.mb = tokenize(mb_tok, texts)
        self.el = tokenize(el_tok, texts)
        self.labels = [int(label) for label in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "mb_input_ids": torch.tensor(self.mb["input_ids"][idx], dtype=torch.long),
            "mb_attention_mask": torch.tensor(self.mb["attention_mask"][idx], dtype=torch.long),
            "el_input_ids": torch.tensor(self.el["input_ids"][idx], dtype=torch.long),
            "el_attention_mask": torch.tensor(self.el["attention_mask"][idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = HybridDataset(train_used_df["model_text"].astype(str).tolist(), train_used_df["label"].tolist())
val_ds = HybridDataset(val_df["model_text"].astype(str).tolist(), val_df["label"].tolist())
test_ds = HybridDataset(test_df["model_text"].astype(str).tolist(), test_df["label"].tolist())


def freeze_lower_encoder_layers(encoder):
    for layer_path, layers in [
        ("encoder.layer", getattr(getattr(encoder, "encoder", None), "layer", None)),
        ("layers", getattr(encoder, "layers", None)),
        ("model.layers", getattr(getattr(encoder, "model", None), "layers", None)),
    ]:
        if layers is not None and len(layers) >= 8:
            # Freeze lower transformer layers to reduce overfitting and computation on short requirement texts.
            for layer in layers[:8]:
                for param in layer.parameters():
                    param.requires_grad = False
            print(f"Frozen lower transformer layers: {layer_path}[0:8]")
            return
    raise ValueError("Could not find transformer layers to freeze.")


def report_trainable_and_frozen_parameters(model):
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    frozen_params = sum(param.numel() for param in model.parameters() if not param.requires_grad)
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters: {frozen_params:,}")

class GatedFusion(nn.Module):
    def __init__(self, n_labels, weights=None):
        super().__init__()
        self.mb = AutoModel.from_pretrained("answerdotai/ModernBERT-base")
        self.el = AutoModel.from_pretrained("google/electra-base-discriminator")
        freeze_lower_encoder_layers(self.mb)
        freeze_lower_encoder_layers(self.el)
        self.gate = nn.Linear(768 * 2, 768)
        self.dropout = nn.Dropout(0.3)
        self.clf = nn.Linear(768, n_labels)
        report_trainable_and_frozen_parameters(self)
        if weights is not None:
            self.register_buffer("weights", weights)
        else:
            self.weights = None

    def forward(self, mb_input_ids=None, mb_attention_mask=None, el_input_ids=None, el_attention_mask=None, labels=None, **kwargs):
        mb_outputs = self.mb(input_ids=mb_input_ids, attention_mask=mb_attention_mask)
        el_outputs = self.el(input_ids=el_input_ids, attention_mask=el_attention_mask)
        h_mb = mb_outputs.last_hidden_state[:, 0, :]
        h_el = el_outputs.last_hidden_state[:, 0, :]
        h_concat = torch.cat([h_mb, h_el], dim=-1)
        g = torch.sigmoid(self.gate(h_concat))
        h_fused = g * h_mb + (1 - g) * h_el
        logits = self.clf(self.dropout(h_fused))
        loss = F.cross_entropy(logits, labels, weight=self.weights) if labels is not None else None
        return {"loss": loss, "logits": logits}

num_labels = len(label_encoder.classes_)
weights = compute_class_weight(class_weight="balanced", classes=np.arange(num_labels), y=train_used_df["label"].astype(int))
weights = torch.tensor(weights, dtype=torch.float)
model = GatedFusion(n_labels=num_labels, weights=weights)

for param in model.parameters():
    if not param.is_contiguous():
        param.data = param.data.contiguous()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": p_w,
        "recall": r_w,
        "f1": f1_w,
        "precision_macro": p_m,
        "recall_macro": r_m,
        "macro_F1": f1_m,
    }

training_args = TrainingArguments(
    output_dir="./modernbert_electra_gated_fusion_" + ("with_augmentation" if True else "without_augmentation"),
    num_train_epochs=30,
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    weight_decay=0.02,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_F1",
    greater_is_better=True,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer.train()

print("Validation Results:")
print(trainer.evaluate(val_ds))
print("\nTesting Results:")
test_results = trainer.predict(test_ds)
test_preds = np.argmax(test_results.predictions, axis=-1)
print(classification_report(test_df["label"].astype(int).tolist(), test_preds, target_names=label_encoder.classes_, zero_division=0))
